# Feature Extraction

Queries AmsterdamUMCdb for the 39 clinical features used by Hilders et al. One row per admission per window.

Input: `data/cohort/sepsis3_cohort.csv`  
Output: `data/features/features_raw.csv`

In [ ]:
import os
import configparser
import psycopg2
import sqlalchemy
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
config = configparser.ConfigParser()
config.read(os.path.join('..', 'AmsterdamUMCdb', 'config.ini'))

output_dir = os.path.join('..', 'data', 'features')
os.makedirs(output_dir, exist_ok=True)

engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{config['psycopg2']['username']}:{config['psycopg2']['password']}"
    f"@{config['psycopg2']['host']}:{config['psycopg2']['port']}/{config['psycopg2']['database']}"
    f"?options=-csearch_path%3Dpublic%20-cclient_encoding%3DWIN1252"
)
con = engine.connect()

cohort = pd.read_csv(os.path.join('..', 'data', 'cohort', 'sepsis3_cohort.csv'))
admission_ids = tuple(cohort['admissionid'].astype(int).tolist())
print(f'Loaded {len(admission_ids)} admissions')

## Query functions


In [ ]:
WINDOW_MINUTES = 240

def assign_window(time_minutes):
    return (time_minutes // WINDOW_MINUTES).astype(int)

# SQL commands taken from AmUMCDB package- amsterdamumcdb/sql/common/gcs.sql, vasopressors_inotropes.sql, heart_rate.sql and sodium.sql

def query_num(itemids, label):
    ids = ', '.join(str(i) for i in itemids)
    sql = f"""
        SELECT n.admissionid, n.value,
               (n.measuredat - a.admittedat) / (1000.0 * 60) AS time_minutes
        FROM numericitems n
        LEFT JOIN admissions a ON n.admissionid = a.admissionid
        WHERE n.itemid IN ({ids})
        AND n.admissionid IN {admission_ids}
        AND (n.measuredat - a.admittedat) >= -(1000*60*30)
    """
    df = pd.read_sql(sqlalchemy.text(sql), con)
    df['feature'] = label
    return df

def query_peep():
    # 8862 = PEEP/CPAP Evita, 8879 = PEEP measured Evita, 12284 = PEEP Set Servo-i
    # 12336 = PEEP Set 2nd Servo-i, 16250 = Zephyros PEEP, 12301/12364 = PEEP total
    sql = f"""
        SELECT n.admissionid, n.value,
               (n.measuredat - a.admittedat) / (1000.0 * 60) AS time_minutes
        FROM numericitems n
        LEFT JOIN admissions a ON n.admissionid = a.admissionid
        WHERE n.itemid IN (8862, 8879, 12284, 12336, 16250, 12301, 12364)
        
        AND n.admissionid IN {admission_ids}
        AND (n.measuredat - a.admittedat) >= -(1000*60*30)
    """
    return pd.read_sql(sqlalchemy.text(sql), con)

def query_gcs(eye_ids, motor_ids, verbal_ids):
    eye_str    = ', '.join(str(i) for i in eye_ids)
    motor_str  = ', '.join(str(i) for i in motor_ids)
    verbal_str = ', '.join(str(i) for i in verbal_ids)
    sql = f"""
        SELECT eyes.admissionid,
               (eyes.measuredat - a.admittedat) / (1000.0 * 60) AS time_minutes,
               CASE eyes.itemid
                   WHEN 6732  THEN 5 - eyes.valueid
                   WHEN 13077 THEN eyes.valueid
                   WHEN 14470 THEN eyes.valueid - 4
                   WHEN 16628 THEN eyes.valueid - 4
                   WHEN 19635 THEN eyes.valueid - 4
                   WHEN 19638 THEN eyes.valueid - 8
               END AS eyes_score,
               CASE motor.itemid
                   WHEN 6734  THEN 7 - motor.valueid
                   WHEN 13072 THEN motor.valueid
                   WHEN 14476 THEN motor.valueid - 6
                   WHEN 16634 THEN motor.valueid - 6
                   WHEN 19636 THEN motor.valueid - 6
                   WHEN 19639 THEN motor.valueid - 12
               END AS motor_score,
               CASE verbal.itemid
                   WHEN 6735  THEN 6 - verbal.valueid
                   WHEN 13066 THEN verbal.valueid
                   WHEN 14482 THEN verbal.valueid - 5
                   WHEN 16640 THEN verbal.valueid - 5
                   WHEN 19637 THEN verbal.valueid - 9
                   WHEN 19640 THEN verbal.valueid - 15
               END AS verbal_score
        FROM listitems eyes
        LEFT JOIN admissions a ON eyes.admissionid = a.admissionid
        LEFT JOIN listitems motor ON
            eyes.admissionid = motor.admissionid AND eyes.measuredat = motor.measuredat
            AND motor.itemid IN ({motor_str})
        LEFT JOIN listitems verbal ON
            eyes.admissionid = verbal.admissionid AND eyes.measuredat = verbal.measuredat
            AND verbal.itemid IN ({verbal_str})
        WHERE eyes.itemid IN ({eye_str})
        AND eyes.admissionid IN {admission_ids}
        AND (eyes.measuredat - a.admittedat) >= 0
    """
    return pd.read_sql(sqlalchemy.text(sql), con)


def query_vasopressors():
    # 7179=dopamine, 7178=dobutamine, 6818=epinephrine, 7229=norepinephrine
    sql = f"""
        SELECT d.admissionid, d.itemid,
               (d.start - a.admittedat) / (1000.0 * 60) AS start_minutes,
               d.dose, d.doserateperkg, d.doseunitid, d.doserateunitid,
               CASE a.weightgroup
                   WHEN '59-'     THEN 55
                   WHEN '60-69'   THEN 65
                   WHEN '70-79'   THEN 75
                   WHEN '80-89'   THEN 85
                   WHEN '90-99'   THEN 95
                   WHEN '100-109' THEN 105
                   WHEN '110+'    THEN 115
                   ELSE 80
               END AS weight
        FROM drugitems d
        LEFT JOIN admissions a ON d.admissionid = a.admissionid
        WHERE d.ordercategoryid = 65
        AND d.itemid IN (7179, 7178, 6818, 7229)
        AND d.rate > 0.1
        AND d.admissionid IN {admission_ids}
    """
    return pd.read_sql(sqlalchemy.text(sql), con)

def query_fluids():
    # 55=crystalloids, 17=colloids, 114=enteral nutrition
    sql = f"""
        SELECT d.admissionid,
               (d.start - a.admittedat) / (1000.0 * 60) AS start_minutes,
               d.fluidin
        FROM drugitems d
        LEFT JOIN admissions a ON d.admissionid = a.admissionid
        WHERE d.ordercategoryid IN (55, 17, 114)
        AND d.fluidin > 0
        AND d.admissionid IN {admission_ids}
    """
    return pd.read_sql(sqlalchemy.text(sql), con)

def query_rrt():
    sql = f"""
        SELECT l.admissionid,
               (l.measuredat - a.admittedat) / (1000.0 * 60) AS time_minutes,
               1 AS rrt_active
        FROM listitems l
        LEFT JOIN admissions a ON l.admissionid = a.admissionid
        WHERE l.itemid = 14846
        AND l.admissionid IN {admission_ids}
        AND (l.measuredat - a.admittedat) >= 0
    """
    return pd.read_sql(sqlalchemy.text(sql), con)

def gamma(row):
    w = row['weight']
    d, uid, ruid, perkg = row['dose'], row['doseunitid'], row['doserateunitid'], row['doserateperkg']
    if perkg == 0 and uid == 11 and ruid == 4: return d / w if w > 0 else d / 80
    elif perkg == 0 and uid == 10 and ruid == 4: return d * 1000 / w if w > 0 else d * 1000 / 80
    elif perkg == 0 and uid == 10 and ruid == 5: return d * 1000 / (w * 60) if w > 0 else d * 1000 / (80 * 60)
    elif perkg == 1 and uid == 11 and ruid == 4: return d
    elif perkg == 1 and uid == 11 and ruid == 5: return d / 60
    return np.nan

print('Functions defined')

## Feature itemid mappings



In [ ]:
# Itemids identified by searching the AmsterdamUMCdb dictionary (`amsterdamumcdb.get_dictionary()`) using Dutch clinical terminology. 
# Multiple itemids per feature cover different measurement systems used across time.
# ClaudeAI. Prompt- "Need to get itemids for each feature.","Give me the english translation to these dutch terms", "Arrange this list in a neat and indented format." 

FEATURES = {
    'heart_rate':  [6640],
    'spo2':        [6709],
    'resp_rate':   [8873, 12266, 8874],
    'sysbp':       [6641, 6678, 8841],
    'meanbp':      [6642, 6679, 8843],
    'diasbp':      [6643, 6680, 8842],
    'tempc':       [8658, 8659, 8662, 13058, 13059, 13060, 13061, 13062, 13063, 13952, 16110],
    'cvp':         [6655],
    'lactate':     [10053, 6837, 9580],
    'creatinine':  [6836, 9941, 14216],
    'bilirubin':   [6813, 9945],
    'platelet':    [6797, 9964, 10409, 14252],
    'hematocrit':  [6777, 11423, 11545],
    'wbc':         [6779, 9965],
    'ph':          [6848, 12310],
    'pao2':        [7433, 9996, 21214],
    'paco2':       [6846, 9990, 21213],
    'sodium':      [6840, 9555, 9924, 10284],
    'potassium':   [6835, 9556, 9927, 10285],
    'glucose':     [6833, 9557, 9947],
    'hemoglobin':  [6778, 9553, 9960, 10286],
    'bicarbonate': [6810, 9992, 9993],
    'chloride':    [6819, 9558, 9930],
    'calcium':     [6817, 9933, 9934],
    'bun':         [6850, 9943],
    'ast':         [6806, 11990],
    'alt':         [6800, 11978],
    'inr':         [6789],
    'albumin':     [6801, 9975],
    'crp':         [6825, 10079],
}
print(f'{len(FEATURES)} numeric features')

## Query and aggregate

In [ ]:
# ClaudeAI. prompt- For querying features in the db, what is the logic for the loop? How do we aggregate the results?
print('Querying numeric features...')
raw_dfs = []
for feat, ids in FEATURES.items():
    df = query_num(ids, feat)
    raw_dfs.append(df)
    print(f'  {feat}: {len(df):,}')

raw = pd.concat(raw_dfs, ignore_index=True)
raw['window'] = assign_window(raw['time_minutes'])
print(f'Total: {len(raw):,}')

In [ ]:
num_agg = (
    raw[raw['value'].notna()]
    .groupby(['admissionid', 'window', 'feature'])['value']
    .mean()
    .reset_index()
    .pivot_table(index=['admissionid', 'window'], columns='feature', values='value')
    .reset_index()
)
num_agg.columns.name = None
print(f'Aggregated: {num_agg.shape}')
num_agg.head()

In [ ]:
# PEEP - MAX per window
peep_raw = query_peep()
peep_raw['window'] = assign_window(peep_raw['time_minutes'])
peep_agg = (
    peep_raw[peep_raw['value'].notna()]
    .groupby(['admissionid', 'window'])['value']
    .max().reset_index()
    .rename(columns={'value': 'peep'})
)
print(f'PEEP: {len(peep_raw):,} measurements, {len(peep_agg):,} windows')

# GCS - min per window
gcs_raw = query_gcs(
    eye_ids   =[6732, 13077, 14470, 16628, 19635, 19638],
    motor_ids =[6734, 13072, 14476, 16634, 19636, 19639],
    verbal_ids=[6735, 13066, 14482, 16640, 19637, 19640]
)
gcs_raw['window'] = assign_window(gcs_raw['time_minutes'])
gcs_agg = (
    gcs_raw.groupby(['admissionid', 'window'])
    .agg(gcs_eye=('eyes_score','min'), gcs_motor=('motor_score','min'), gcs_verbal=('verbal_score','min'))
    .reset_index()
)
print(f'GCS: {len(gcs_raw):,} measurements, {len(gcs_agg):,} windows')

# Vasopressors - max norepinephrine-equivalent per window
vaso_raw = query_vasopressors()
vaso_raw['doserateperkg'] = vaso_raw['doserateperkg'].apply(
    lambda x: 0 if str(x) in ("b'0'", '0', 'False') else 1
)
vaso_raw['gamma_val'] = vaso_raw.apply(gamma, axis=1)
NE = {7229: 1.0, 6818: 1.0, 7179: 0.01, 7178: 0.01}
vaso_raw['ne_equiv'] = vaso_raw['gamma_val'] * vaso_raw['itemid'].map(NE)
vaso_raw['window'] = assign_window(vaso_raw['start_minutes'].clip(lower=0))
vaso_agg = (
    vaso_raw.groupby(['admissionid', 'window'])['ne_equiv']
    .max().reset_index()
    .rename(columns={'ne_equiv': 'vaso_rate'})
)
print(f'Vasopressors: {len(vaso_raw):,} orders, {len(vaso_agg):,} windows')

# IV fluids - sum per window
fluid_raw = query_fluids()
fluid_raw['window'] = assign_window(fluid_raw['start_minutes'].clip(lower=0))
fluid_agg = (
    fluid_raw[fluid_raw['fluidin'].notna()]
    .groupby(['admissionid', 'window'])['fluidin']
    .sum().reset_index()
    .rename(columns={'fluidin': 'fluid_window'})
)
print(f'Fluids: {len(fluid_raw):,} orders, {len(fluid_agg):,} windows')

# RRT - binary, max per window
rrt_raw = query_rrt()
rrt_raw['window'] = assign_window(rrt_raw['time_minutes'])
rrt_agg = (
    rrt_raw.groupby(['admissionid', 'window'])['rrt_active']
    .max().reset_index()
    .rename(columns={'rrt_active': 'rrt'})
)
print(f'RRT: {len(rrt_raw):,} records, {len(rrt_agg):,} windows')

## Build time grid and merge



In [ ]:
# ClaudeAI. Prompt- Hilders et al. uses a time grid for patients per window to forward fill the missing values. How do i setup the time grid?
adm_df = pd.read_sql(
    sqlalchemy.text(f"SELECT admissionid, admittedat, dischargedat FROM admissions WHERE admissionid IN {admission_ids}"),
    con
)
adm_df['los_min'] = (adm_df['dischargedat'] - adm_df['admittedat']) / (1000 * 60)
adm_df['n_windows'] = (adm_df['los_min'] / WINDOW_MINUTES).fillna(1).apply(np.ceil).astype(int).clip(lower=1)

grid = pd.concat([
    pd.DataFrame({'admissionid': row['admissionid'], 'window': range(int(row['n_windows']))})
    for _, row in adm_df.iterrows()
], ignore_index=True)

grid['start_time'] = (
    adm_df.set_index('admissionid').loc[grid['admissionid'], 'admittedat'].values
    + grid['window'] * WINDOW_MINUTES * 60 * 1000
)
print(f'Time grid: {len(grid):,} rows ({grid.admissionid.nunique()} admissions)')

In [ ]:
features = grid.copy()
for df_m, name in [
    (num_agg, 'numeric'), (peep_agg, 'PEEP'), (gcs_agg, 'GCS'),
    (vaso_agg, 'vasopressors'), (fluid_agg, 'fluids'), (rrt_agg, 'RRT'),
]:
    features = features.merge(df_m, on=['admissionid', 'window'], how='left')
    print(f'  merged {name}')

demo_cols = [c for c in ['admissionid', 'gender', 'agegroup', 'weightgroup'] if c in cohort.columns]
features = features.merge(cohort[demo_cols].drop_duplicates(), on='admissionid', how='left')
features['gender'] = np.where(features['gender'] == 'Man', 1, 0)
features['start_time'] = pd.to_datetime(features['start_time'], unit='ms')
print(f'Final: {features.shape}')

## Missingness summary

High missingness is expected for infrequently drawn labs. Forward-fill in the next notebook propagates values forward up to 24h.

In [ ]:
feat_cols = [c for c in features.columns if c not in ['admissionid', 'window', 'start_time', 'agegroup', 'weightgroup', 'gender']]
miss = (features[feat_cols].isna().sum() / len(features)).round(3).sort_values(ascending=False)
print(miss.to_string())

In [ ]:
features.to_csv(os.path.join(output_dir, 'features_raw.csv'), index=False)
print(f'Saved {len(features):,} rows ({features.admissionid.nunique()} admissions)')
con.close()